# פרויקט סיום: איסוף והעשרת נתונים - IMDb & Wikipedia

**חברי הצוות:**
* שיראל חדד - [212546017]

**קישור למאגר הפרויקט ב-GitHub:**
[לינק לגיט שלך כאן]

In [1]:
import sys
!{sys.executable} -m pip install wikipedia

Defaulting to user installation because normal site-packages is not writeable


In [35]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import wikipedia
import time
import warnings
import urllib.parse

In [59]:
# העמודות שרלוונטיות לנו 
basics_cols = ['tconst', 'titleType', 'primaryTitle', 'startYear', 'genres', 'runtimeMinutes']
df_basics = pd.read_csv('data/title.basics.tsv.gz', sep='\t', usecols=basics_cols, low_memory=False)
#סינון רק של הסרטים 
df_basics = df_basics[df_basics['titleType'] == 'movie'].copy()
df_basics = df_basics.drop(columns=['titleType'])
# המרה למספרים 
df_basics['startYear'] = pd.to_numeric(df_basics['startYear'], errors='coerce')
df_basics['runtimeMinutes'] = pd.to_numeric(df_basics['runtimeMinutes'], errors='coerce')

# סינונים לפי הדרישות 
mask = (df_basics['startYear'] <= 2024) & \
       (df_basics['primaryTitle'].str.startswith(('W','X'), na=False)) & \
       (df_basics['runtimeMinutes'].between(60, 300))

df_movies = df_basics[mask].copy()

# מסננים רק את הדירוגים של הסרטים שלנו
relevant_tconsts = set(df_movies['tconst'])

ratings_chunks = pd.read_csv('data/title.ratings.tsv.gz', sep='\t', chunksize=500000)
filtered_ratings_list = [chunk[chunk['tconst'].isin(relevant_tconsts)] for chunk in ratings_chunks]
df_ratings_filtered = pd.concat(filtered_ratings_list)

final_df = pd.merge(df_movies, df_ratings_filtered, on='tconst', how='inner')


# לוקחים את ה5000 סרטים שיש להם הכי הרבה דירוגים 
if len(final_df) > 5000:
    final_df = final_df.sort_values(by='numVotes', ascending=False).head(5000).reset_index(drop=True)

In [60]:
relevant_movies = set(final_df['tconst'])
# טעינת עמודות רלוונטיות בלבד
principals_cols = ['tconst', 'nconst', 'category', 'ordering']
chunks = pd.read_csv('data/title.principals.tsv.gz', sep='\t', usecols=principals_cols, chunksize=500000)
filtered_principals = []

for chunk in chunks:
    mask = (chunk['tconst'].isin(relevant_movies)) & \
           (chunk['category'].isin(['actor', 'actress']))
    
    filtered_chunk = chunk[mask]
    if not filtered_chunk.empty:
        filtered_principals.append(filtered_chunk)

# מאחדים רק את הנתחים הרלוונטיים
df_actors = pd.concat(filtered_principals)
# מיון ושליפת 5 הראשונים
df_actors = df_actors.sort_values(by=['tconst', 'ordering'])
top_5_actors = df_actors.groupby('tconst').head(5)

# קיבוץ לרשימה
actors_list = top_5_actors.groupby('tconst')['nconst'].apply(list).reset_index()
actors_list.columns = ['tconst', 'lead_actors_ids']

final_df = pd.merge(final_df, actors_list, on='tconst', how='left')

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids
0,tt0910970,WALL·E,2008.0,98.0,"Adventure,Animation,Family",8.4,1319342,"[nm0123785, nm0123785, nm2264184, nm0307531, n..."
1,tt2582802,Whiplash,2014.0,106.0,"Drama,Music",8.5,1152780,"[nm1886602, nm0799777, nm2552034, nm0001663, n..."
2,tt1877832,X-Men: Days of Future Past,2014.0,132.0,"Action,Adventure,Sci-Fi",7.9,787522,"[nm0001772, nm0005212, nm0413168, nm0413168, n..."
3,tt0816711,World War Z,2013.0,116.0,"Action,Adventure,Horror",7.0,782471,"[nm0000093, nm0257969, nm2020146, nm0197647, n..."
4,tt1270798,X-Men: First Class,2011.0,131.0,"Action,Sci-Fi",7.7,763614,"[nm0564215, nm1055413, nm2225369, nm2225369, n..."
5,tt0451279,Wonder Woman,2017.0,141.0,"Action,Adventure,Fantasy",7.3,735214,"[nm2933757, nm1517976, nm0000705, nm0205063, n..."
6,tt0120903,X-Men,2000.0,104.0,"Action,Adventure,Sci-Fi",7.3,691388,"[nm0001772, nm0413168, nm0413168, nm0005212, n..."
7,tt0290334,X2: X-Men United,2003.0,134.0,"Action,Sci-Fi,Thriller",7.4,611446,"[nm0001772, nm0413168, nm0413168, nm0000932, n..."
8,tt0409459,Watchmen,2009.0,162.0,"Action,Drama,Mystery",7.6,610595,"[nm0355097, nm0933940, nm0933940, nm0001303, n..."
9,tt0376994,X-Men: The Last Stand,2006.0,104.0,"Action,Adventure,Sci-Fi",6.6,571956,"[nm0001772, nm0413168, nm0413168, nm0000932, n..."


In [61]:
def parse_money_to_millions(money_str):
    #מחלצת מספרים, מחשבת ממוצע ומחזירה ערך ביחידות של מיליוני דולרים
    if not money_str or pd.isna(money_str) or money_str == "None":
        return None
    
    # ניקוי בסיסי של סימנים שעלולים להפריע לזיהוי המספרים
    clean_str = money_str.replace(',', '').replace('$', '')
    
    # מציאת כל המספרים 
    numbers = re.findall(r'\d+\.?\d*', clean_str)
    
    if not numbers:
        return None
        
    # המרה ל-float
    values = [float(n) for n in numbers]
    
    # חישוב הממוצע אם מדובר בטווח 
    base_value = sum(values[:2]) / 2 if len(values) >= 2 else values[0]
    
    # קביעת המכפיל לפי המילה המופיעה בטקסט
    multiplier = 1
    if 'million' in money_str.lower():
        multiplier = 1  
    elif 'billion' in money_str.lower():
        multiplier = 1000ן
    else:
        #נחלק במיליון כדי להגיע ליחידות הנכונות
        multiplier = 0.000001
        
    return round(base_value * multiplier, 2)
# השתקת אזהרות ה-Parser של BeautifulSoup
warnings.filterwarnings("ignore", category=GuessedAtParserWarning)

# הגדרת "זהות" הדפדפן
BROWSER_AGENT = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
}

def format_wiki_content(raw_html_text):
    #מנקה הערות שוליים, רווחים וסימנים מיותרים מהטקסט של ויקיפדיה
    if not raw_html_text:
        return None
    # הסרת סוגריים מרובעים של הערות שוליים כמו [1]
    text_no_refs = re.sub(r'\[.*?\]', '', raw_html_text)
    # החלפת ירידות שורה בפסיקים וניקוי רווחים כפולים
    cleaned = text_no_refs.replace('\n', ', ').strip()
    return re.sub(r',\s*,', ',', cleaned).strip(' ,')

def fetch_movie_details(title, target_year, only_money=False):
    #פונקציה שסורקת את ויקיפדיה לפי שם סרט ושנה
    # הכנת השנה והשם לכתובת URL
    movie_year = str(int(float(target_year))) if pd.notna(target_year) else ""
    web_safe_title = urllib.parse.quote(title.replace(' ', '_'))
    
    # יצירת סדר עדיפויות לכתובות לחיפוש
    possible_links = []
    if movie_year:
        possible_links.append(f"https://en.wikipedia.org/wiki/{web_safe_title}_({movie_year}_film)")
    possible_links.append(f"https://en.wikipedia.org/wiki/{web_safe_title}_(film)")
    possible_links.append(f"https://en.wikipedia.org/wiki/{web_safe_title}")

    for link in possible_links:
        try:
            res = requests.get(link, headers=BROWSER_AGENT, timeout=7)
            if res.status_code != 200:
                continue
                
            page_content = BeautifulSoup(res.content, 'html.parser')
            
            # אם זה דף פירושונים - נדלג
            if page_content.find(class_="disambigbox"):
                continue

            movie_table = page_content.find('table', class_=lambda x: x and 'infobox' in x)
            if not movie_table:
                continue
                
            # אימות שהסרט אכן מהשנה הנכונה 
            table_raw_text = movie_table.get_text()
            if movie_year:
                # בדיקה אם השנה מופיעה בטבלה
                valid_years = [movie_year, str(int(movie_year)-1), str(int(movie_year)+1)]
                if not any(y in table_raw_text for y in valid_years):
                    continue

            # הכנת מילון הנתונים לשליפה
            extracted_info = {'Language': None, 'Country': None, 'budget': None, 'BoxOffice': None, 'summary': None}
            
            # מעבר על שורות הטבלה
            for row in movie_table.find_all('tr'):
                header_cell = row.find('th')
                data_cell = row.find('td')
                
                if header_cell and data_cell:
                    label = header_cell.get_text().lower()
                    val = format_wiki_content(data_cell.get_text(separator=', '))
                    
                    # בתוך הלולאה של ה-Infobox, איפה שכתוב:
                    if 'budget' in label:
                        raw_val = format_wiki_content(data_cell.get_text(separator=', '))
                        extracted_info['budget'] = parse_money_to_millions(raw_val) 
                    elif 'box office' in label:
                        raw_val = format_wiki_content(data_cell.get_text(separator=', '))
                        extracted_info['BoxOffice'] = parse_money_to_millions(raw_val) 
                                        
                    # אם לא ביקשנו רק נתונים כספיים, נשלוף גם שפה ומדינה
                    if not only_money:
                        if 'language' in label:
                            extracted_info['Language'] = val
                        elif 'countr' in label or 'origin' in label:
                            extracted_info['Country'] = val
            
            # שליפת התקציר (Plot) מהדף
            if not only_money:
                plot_tag = page_content.find(id=lambda x: x and x.lower() in ['plot', 'synopsis', 'storyline'])
                if plot_tag and plot_tag.parent:
                    paragraphs = []
                    next_node = plot_tag.parent.find_next_sibling()
                    # איסוף פסקאות עד שמגיעים לכותרת הבאה (H2 או H3)
                    while next_node and next_node.name not in ['h2', 'h3']:
                        if next_node.name == 'p':
                            paragraphs.append(next_node.get_text().strip())
                        next_node = next_node.find_next_sibling()
                    extracted_info['summary'] = " ".join(paragraphs) if paragraphs else None

            return extracted_info
            
        except:
            continue
            
    return {'Language': None, 'Country': None, 'budget': None, 'BoxOffice': None, 'summary': None}

In [62]:

for column in ['Language', 'Country', 'budget', 'BoxOffice', 'plot']:
    final_df[column] = None

print("מתחיל איסוף נתונים מוויקיפדיה...")

for idx, row in final_df.iterrows():
    movie_info = fetch_movie_details(row['primaryTitle'], row['startYear'])
    
    # עדכון ישיר של השורה בטבלה המקורית
    final_df.at[idx, 'Language'] = movie_info['Language']
    final_df.at[idx, 'Country'] = movie_info['Country']
    final_df.at[idx, 'budget'] = movie_info['budget']
    final_df.at[idx, 'BoxOffice'] = movie_info['BoxOffice']
    final_df.at[idx, 'plot'] = movie_info['summary']
    
    # הדפסת התקדמות בכל 100 סרטים
    current_num = idx + 1
    if current_num % 100 == 0:
        print(f"התקדמות: {current_num}/{len(final_df)} סרטים...")
    
    # שמירה אוטומטית כול 500 שורות 
    if current_num % 500 == 0:
        print(f"מבצע שמירת גיבוי בסרט ה-{current_num}...")
        final_df.to_csv('movies_data_checkpoint.csv', index=False, encoding='utf-8-sig')
    
    # השהיה קלה למניעת חסימות מוויקיפדיה
    time.sleep(0.5)

print("הסתיים איסוף הנתונים!")

#שמירה סופית של הקובץ המועשר
final_df.to_csv('final_movies_dataset_with_wiki.csv', index=False, encoding='utf-8-sig')


מתחיל איסוף נתונים מוויקיפדיה...
התקדמות: 100/5000 סרטים...
התקדמות: 200/5000 סרטים...
התקדמות: 300/5000 סרטים...
התקדמות: 400/5000 סרטים...
התקדמות: 500/5000 סרטים...
מבצע שמירת גיבוי בסרט ה-500...
התקדמות: 600/5000 סרטים...
התקדמות: 700/5000 סרטים...
התקדמות: 800/5000 סרטים...
התקדמות: 900/5000 סרטים...
התקדמות: 1000/5000 סרטים...
מבצע שמירת גיבוי בסרט ה-1000...
התקדמות: 1100/5000 סרטים...
התקדמות: 1200/5000 סרטים...
התקדמות: 1300/5000 סרטים...
התקדמות: 1400/5000 סרטים...
התקדמות: 1500/5000 סרטים...
מבצע שמירת גיבוי בסרט ה-1500...
התקדמות: 1600/5000 סרטים...
התקדמות: 1700/5000 סרטים...
התקדמות: 1800/5000 סרטים...
התקדמות: 1900/5000 סרטים...
התקדמות: 2000/5000 סרטים...
מבצע שמירת גיבוי בסרט ה-2000...
התקדמות: 2100/5000 סרטים...
התקדמות: 2200/5000 סרטים...
התקדמות: 2300/5000 סרטים...
התקדמות: 2400/5000 סרטים...
התקדמות: 2500/5000 סרטים...
מבצע שמירת גיבוי בסרט ה-2500...
התקדמות: 2600/5000 סרטים...
התקדמות: 2700/5000 סרטים...
התקדמות: 2800/5000 סרטים...
התקדמות: 2900/5000 סרטים...
התקדמ